In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(
            to right,
            limegreen,
            deepskyblue,
            limegreen
        );
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * **02-05. Tools**
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-06. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-07. **AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`**
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---
# AccuWeather Agent with `ComputerTool`
* Uses `ComputerTool` to automate browser interactions via Playwright ― a Microsoft open-source browser automation tool
* Opens AccuWeather website and searches for user-specified locations
* Agent "sees" screenshots and controls mouse, keyboard and scrolling
* Presents matching locations when AccuWeather shows choices
* Selects the user-chosen location through the browser
* Reads visible weather tabs and lets user choose one
* Supports follow-up searches for new locations
* Browser state persists across turns while computer context is open
* Conversation state preserves user choices and prior agent messages

**Reference documentation**
* https://developers.openai.com/api/docs/guides/tools-computer-use

---

## Agents SDK `Computer` vs `AsyncComputer` Interfaces

* `ComputerTool` requires a subclass of `Computer` or `AsyncComputer`
    | | `Computer` | `AsyncComputer` |
    |---|---|---|
    | Method style | `def click(...)` | `async def click(...)` |
    | Suited for | Sync libraries (pyautogui, PIL, …) | Async libraries (Playwright, …) |
* Both define the same interface 
* Choose `AsyncComputer` whenever the library uses `async`/`await`
---

## Playwright Microsoft Open-Source Browser Automation Library
* https://playwright.dev/python/
* Can drive Chromium, Firefox, and WebKit browsers programmatically 
* OS mouse and keyboard are **not** used 
    * Playwright sends commands directly to browser 
    * User can use other apps in parallel
* `ComputerTool` also supports automating macOS, Windows and Linux

---

## `LocalBrowserComputer` Class
* [Open browser_computer.py](./source/browser_computer.py) to view source
* Written by a combo of OpenAI Codex and Claude Code
* Implements `AsyncComputer` using Playwright
    * `__aenter__` / `__aexit__` launch and close the browser
    * `screenshot()` captures the viewport as a base64 PNG — sent to the model
    * Action methods (`click`, `type`, `scroll`, …) translate SDK calls to Playwright
    * `KEY_MAP` converts model-facing key names (e.g. `"ctrl"`) to Playwright names (e.g. `"Control"`)
    * `hold_keys()` holds modifier keys during mouse actions (e.g. Ctrl+click)


In [ ]:
from source.browser_computer import LocalBrowserComputer

## Imports and Constants

* `BROWSER_CHANNEL = 'chrome'` uses your locally installed Google Chrome
    * Requires Google Chrome to be installed; no extra Playwright download needed
    * Change to `None` to use Playwright's bundled Chromium instead
        * `python -m playwright install chromium` to download it first
* Browser launches visibly by default for classroom demonstration
* Type `exit` to close browser and end loop

---


## Weather Browser Agent
* Agent uses only the browser computer tool
* It must present ambiguous location choices instead of guessing
* It must list visible weather tabs after a location page loads
* User can then choose a tab or enter another location

---

In [ ]:
from IPython.display import Markdown, display
from agents import Agent, ComputerTool, ModelSettings, Runner, trace

def create_weather_agent(computer: LocalBrowserComputer) -> Agent:
    """Create the browser-based AccuWeather agent."""

    # ComputerTool wraps an object of class LocalBrowserComputer. 
    # The model "sees" screenshots from this object and sends back browser actions.
    return Agent(
        name='AccuWeather Browser Agent',
        model='gpt-5.5',
        model_settings=ModelSettings(
            tool_choice='computer', # tell model to use ComputerTool 
            reasoning={'effort': 'low'},
            verbosity='low'
        ),
        instructions="""Use the browser computer tool to interact with
            AccuWeather. Start at accuweather.com. Search for the user's
            requested location. If AccuWeather shows multiple matching
            locations, do not choose. Read the visible choices and present
            a numbered list to the user. When the user chooses one, click
            that result. Once a location page is displayed, summarize the
            current weather and read the available tabs or navigation
            choices. Present those tabs to the user. If the user selects
            one, click it and summarize what appears. If the user enters
            another location, use the page search field to search for it.
            Keep responses concise.""",
        tools=[ComputerTool(computer=computer)]
    )

def first_turn_prompt(location: str) -> str:
    """Build the first user request."""

    # first turn starts from AccuWeather's home page 
    return f"""
    Search AccuWeather for this location:
    {location}

    If AccuWeather displays multiple matching locations, list them for
    the user and wait for selection. If a single location opens, summarize
    current weather and list available weather tabs.
    """

def follow_up_prompt(user_text: str) -> str:
    """Build follow-up requests after browser state exists."""

    # Later turns reuse the same browser page and the conversation state.
    # Agent decides whether the user typed a location, a listed choice
    # or a weather-tab name.
    return f"""
    The user responded:
    {user_text}

    If this is a numbered or named location choice, select that location.
    If this is a tab name, click that weather tab. If this is a new
    location, use AccuWeather's search field to search for it. After the
    page updates, summarize what is shown and list any available tabs.
    """

## Run a Multi-turn AccuWeather Browser Session

* Type a location, such as `Springfield` or `Boston, MA`
* If multiple locations are listed, type the number or name to choose
* If tabs are listed, type a tab name such as `Hourly` or `Radar`
* Type another location to search again
* Type `exit` to close the browser


In [ ]:
# 'chrome' uses your locally installed Google Chrome (recommended for demos);
# set to None to use Playwright's bundled Chromium instead
BROWSER_CHANNEL = 'chrome'

# False opens a visible browser window for live classroom presentation;
# set to True only to run unattended, but carefully consider security implications
HEADLESS = False

async with LocalBrowserComputer(
    start_url='https://www.accuweather.com/', 
    channel=BROWSER_CHANNEL, 
    headless=HEADLESS
) as computer:
    agent = create_weather_agent(computer)

    previous_response_id = None
    first_turn = True

    display(Markdown(
        'Enter a location, a listed choice, a tab name or `exit`.'))

    with trace('02-05-07-computer-tool-accuweather'):
        while True:
            user_text = input('\nLocation, choice, tab or exit: ').strip()

            if user_text.lower() == 'exit':
                display(Markdown('**AccuWeather demo ended**'))
                break

            if first_turn:
                prompt = first_turn_prompt(user_text) 
                first_turn = False
            else: 
                prompt = follow_up_prompt(user_text)

            display(Markdown('**Agent is using the browser...**'))

            result = await Runner.run(
                agent,
                prompt,
                previous_response_id=previous_response_id
            )

            display(Markdown(result.final_output))
            previous_response_id = result.last_response_id

---
## Notes

* The agent may need to close cookie notices or overlays
* This example intentionally uses the public website, not a weather API
* For deterministic weather data, use an API-based tool instead

---

## References

* [OpenAI Agents SDK: Tools](https://openai.github.io/openai-agents-python/tools/)
* [OpenAI Agents SDK: Tracing](https://openai.github.io/openai-agents-python/tracing/)
  [OpenAI Computer Use Guide](https://developers.openai.com/api/docs/guides/tools-computer-use)
* [SDK example: computer_use.py](https://github.com/openai/openai-agents-python/blob/main/examples/tools/computer_use.py)
* [Playwright Python](https://playwright.dev/python/)

---

## Controlling the OS — PyAutoGUI
* Cross-platform library for OS-level mouse/keyboard control

### PyAutoGUI Docs
* [Overview](https://pyautogui.readthedocs.io/en/latest/)
* [Quickstart / cheat sheet](https://pyautogui.readthedocs.io/en/latest/quickstart.html)
* [Mouse control](https://pyautogui.readthedocs.io/en/latest/mouse.html)
* [Keyboard control](https://pyautogui.readthedocs.io/en/latest/keyboard.html)

---

© 2026 by Deitel & Associates, Inc. All Rights Reserved.